# What a CNN sees
Open VGG16 (trained on ImageNet): look at its first-layer filters (and ResNet50's larger ones), at the feature maps one
photo produces layer by layer, at how sparse those maps get with depth, and at which image patches excite single
feature maps in deeper layers (the method of Zeiler and Fergus 2014).

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"          # hide TensorFlow's start-up messages
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"  # take GPU memory only as needed
os.environ["TF_CUDNN_WORKSPACE_LIMIT_IN_MB"] = "256"   # the GPU is shared; keep scratch memory small
import warnings
warnings.filterwarnings("ignore", category=UserWarning)   # Keras notices about tf.data
from pathlib import Path
import numpy as np
import pandas as pd
import tensorflow as tf
import keras
from keras import layers

tf.get_logger().setLevel("ERROR")
HERE = Path.cwd().parent if Path.cwd().name == ".logs" else Path.cwd()   # the Note folder, also from .logs/
(HERE / "data").mkdir(exist_ok=True)
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| GPU:", bool(tf.config.list_physical_devices("GPU")))

In [ ]:
import urllib.request
PHOTOS = Path.home() / "datasets" / "photos"

def fetch(url):
    """Download a photo once (Wikimedia asks for a User-Agent) and return its local path."""
    PHOTOS.mkdir(parents=True, exist_ok=True)
    path = PHOTOS / url.split("/")[-1]
    if not path.exists():
        request = urllib.request.Request(url, headers={"User-Agent": "campusx-notes/1.0 (study notes)"})
        path.write_bytes(urllib.request.urlopen(request).read())
    return path

In [ ]:
from keras.applications.vgg16 import VGG16, preprocess_input
vgg = VGG16(weights="imagenet")                  # the full network, with its dense layers
for i, layer in enumerate(vgg.layers):
    print(i, layer.name, layer.output.shape)

In [ ]:
conv_ids = [i for i, layer in enumerate(vgg.layers) if "conv" in layer.name]
print("convolution layers at positions", conv_ids, "->", len(conv_ids), "convolution layers + 3 dense = 16 layers with weights")

## 1. The filters of the first layer

In [ ]:
filters, biases = vgg.layers[1].get_weights()     # block1_conv1
print("VGG16 first layer: filters", filters.shape, "biases", biases.shape)
f_min, f_max = filters.min(), filters.max()
scaled = (filters - f_min) / (f_max - f_min)      # 0..1, so a filter can be shown as a tiny colour image
np.save(HERE / "data" / "vgg16_filters.npy", scaled.astype(np.float16))

from keras.applications.resnet50 import ResNet50
resnet = ResNet50(weights="imagenet", include_top=False)
w = resnet.get_layer("conv1_conv").get_weights()[0]
print("ResNet50 first layer:", w.shape)
np.save(HERE / "data" / "resnet50_filters.npy", ((w - w.min()) / (w.max() - w.min())).astype(np.float16))

## 2. Feature maps of one photo, layer by layer

In [ ]:
path = fetch("https://upload.wikimedia.org/wikipedia/commons/b/bb/Kittyply_edit1.jpg")
img = keras.utils.load_img(path, target_size=(224, 224))
x = preprocess_input(np.expand_dims(keras.utils.img_to_array(img), 0))
img.save(HERE / "data" / "kitten_224.jpg", quality=85)

layer_ids = [1, 2, 5, 9, 13, 17]                 # one or two convolution layers per block
probe = keras.Model(inputs=vgg.inputs, outputs=[vgg.layers[i].output for i in layer_ids])
maps = probe.predict(x, verbose=0)
saved = {}
for i, m in zip(layer_ids, maps):
    m = m[0]                                     # (height, width, channels)
    first8 = m[..., :8]
    side = min(m.shape[0], 112)
    small = tf.image.resize(first8, (side, side)).numpy()
    lo, hi = small.min(axis=(0, 1)), small.max(axis=(0, 1))
    saved[f"layer{i}"] = (255 * (small - lo) / np.maximum(hi - lo, 1e-6)).astype(np.uint8)
    print(f"layer {i:2d} {vgg.layers[i].name:13s} maps {m.shape}, blank maps (all zero): {(m.max(axis=(0, 1)) == 0).mean():.0%}")
np.savez_compressed(HERE / "data" / "feature_maps.npz", **saved)

## 3. Sparsity with depth, over 500 photos of cats and dogs

In [ ]:
# The Kaggle / Microsoft cats-vs-dogs photos: 12,500 cats and 12,500 dogs, one folder per class.
DATA = Path.home() / "datasets" / "PetImages"
URL = ("https://download.microsoft.com/download/3/E/1/"
       "3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip")
if not DATA.exists():
    zip_path = keras.utils.get_file("kagglecatsanddogs_5340.zip", URL, cache_dir=Path.home(), cache_subdir="datasets")
    import shutil
    shutil.unpack_archive(zip_path, DATA.parent)
# Some files are damaged or are not JPEGs at all; delete them, as the official Keras example does.
removed = 0
for path in DATA.glob("*/*"):
    with open(path, "rb") as f:
        is_jpeg = b"JFIF" in f.peek(10)
    if not is_jpeg:
        path.unlink()
        removed += 1
counts = {d.name: len(list(d.iterdir())) for d in sorted(DATA.iterdir())}
print("removed now:", removed, "| images per class:", counts, "| total:", sum(counts.values()))

In [ ]:
# A small subset, as when data is scarce: 1,000 cats and 1,000 dogs for training,
# 500 + 500 for validation and 500 + 500 for testing, picked at random once (seed 0).
# Symbolic links put them in the folder layout that image_dataset_from_directory expects.
SMALL = Path.home() / "datasets" / "cats_vs_dogs_small"
if not SMALL.exists():
    rng = np.random.default_rng(0)
    for cls in ("Cat", "Dog"):
        files = sorted(DATA.glob(f"{cls}/*.jpg"), key=lambda p: int(p.stem))
        pick = rng.choice(len(files), 2000, replace=False)
        for split, idx in (("train", pick[:1000]), ("validation", pick[1000:1500]), ("test", pick[1500:])):
            folder = SMALL / split / cls.lower()
            folder.mkdir(parents=True, exist_ok=True)
            for i in idx:
                (folder / files[i].name).symlink_to(files[i])
print({split: sum(1 for _ in (SMALL / split).glob("*/*")) for split in ("train", "validation", "test")})

In [ ]:
files = sorted((SMALL / "train").glob("*/*.jpg"))[::4]          # 500 photos, both classes
def load_batch(paths):
    return preprocess_input(np.stack([keras.utils.img_to_array(keras.utils.load_img(p, target_size=(224, 224)))
                                      for p in paths]))
probe_all = keras.Model(inputs=vgg.inputs, outputs=[vgg.layers[i].output for i in conv_ids])
zero_frac = np.zeros(len(conv_ids)); blank_frac = np.zeros(len(conv_ids))
for start in range(0, len(files), 50):
    outs = probe_all.predict(load_batch(files[start:start + 50]), verbose=0)
    for k, o in enumerate(outs):
        zero_frac[k] += (o == 0).mean() * len(o)
        blank_frac[k] += (o.max(axis=(1, 2)) == 0).mean() * len(o)
sparsity = pd.DataFrame(dict(position=conv_ids, layer=[vgg.layers[i].name for i in conv_ids],
                             zero_values=zero_frac / len(files), blank_maps=blank_frac / len(files))).round(4)
sparsity.to_csv(HERE / "data" / "sparsity.csv", index=False)
print(sparsity.to_string(index=False))

## 4. What excites one feature map deep in the network
For three layers, take the two feature maps (channels) that the kitten photo excites most. Then search 1,000 photos
of cats and dogs for the 6 photos where each map responds most strongly, and cut out the patch of the photo that
the map's strongest value depends on (its receptive field).

In [ ]:
probe_deep = {name: keras.Model(vgg.inputs, vgg.get_layer(name).output)
              for name in ("block3_conv3", "block4_conv3", "block5_conv3")}
# receptive field of one value: 40, 92 and 196 pixels (3 x 3 convolutions and 2 x 2 pooling, from the input)
patch_size = {"block3_conv3": 40, "block4_conv3": 92, "block5_conv3": 196}
chosen = {}
for name, model in probe_deep.items():
    m = model.predict(x, verbose=0)[0]
    chosen[name] = list(np.argsort(-m.max(axis=(0, 1)))[:2])
print("channels chosen (most excited by the kitten):", {k: [int(c) for c in v] for k, v in chosen.items()})

pool = sorted((SMALL / "train").glob("*/*.jpg"))[::2]          # 1,000 photos
best = {(n, c): [] for n in chosen for c in chosen[n]}         # (score, file, row, col, map size)
for start in range(0, len(pool), 50):
    paths = pool[start:start + 50]
    xb = load_batch(paths)
    for name, model in probe_deep.items():
        out = model.predict(xb, verbose=0)
        for c in chosen[name]:
            a = out[..., c]
            for k in range(len(paths)):
                r, col = np.unravel_index(a[k].argmax(), a[k].shape)
                best[(name, c)].append((float(a[k].max()), paths[k], r, col, a.shape[1]))

In [ ]:
from PIL import Image
TILE = 64
grid = Image.new("RGB", (6 * (TILE + 4), 6 * (TILE + 4)), "white")
rows = []
for row, (key, hits) in enumerate(best.items()):
    name, c = key
    hits.sort(key=lambda h: -h[0])
    for col, (score, p, r, cc, size) in enumerate(hits[:6]):
        photo = np.array(keras.utils.load_img(p, target_size=(224, 224)))
        s = patch_size[name]
        cy, cx = (r + 0.5) * 224 / size, (cc + 0.5) * 224 / size
        y0 = int(np.clip(cy - s / 2, 0, 224 - s)); x0 = int(np.clip(cx - s / 2, 0, 224 - s))
        crop = Image.fromarray(photo[y0:y0 + s, x0:x0 + s]).resize((TILE, TILE))
        grid.paste(crop, (col * (TILE + 4), row * (TILE + 4)))
        rows.append(dict(layer=name, channel=int(c), rank=col + 1, activation=round(score, 1), file=p.parent.name + "/" + p.name))
grid.save(HERE / "data" / "top_patches.jpg", quality=85)
top = pd.DataFrame(rows)
top.to_csv(HERE / "data" / "top_patches.csv", index=False)
print(top.groupby(["layer", "channel"]).file.apply(lambda f: ", ".join(f)).to_string())

## 5. Is there a 'person' class in ImageNet?

In [ ]:
import json
index = json.load(open(keras.utils.get_file("imagenet_class_index.json",
        "https://storage.googleapis.com/download.tensorflow.org/data/imagenet_class_index.json")))
names = [index[str(i)][1] for i in range(1000)]
for word in ("person", "human", "people", "man", "woman", "face"):
    print(f"{word:7s}", [n for n in names if word in n.lower().split("_")])